In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from skorch import NeuralNetClassifier
from sklearn.model_selection import GridSearchCV

import joblib
from joblib import Parallel, delayed
import logging 

In [2]:
dataset_names = ['adult', 'compas', 'diabetes', 'german', 'heloc', 'independent']

In [3]:
datasets = []
for dataset in dataset_names:
    # load the dataset, split into input (X) and output (y) variables
    X = pd.read_csv('./data/' + dataset +'/X_train_prep.csv', delimiter=',')
    y = pd.read_csv('./data/' + dataset +'/y_train.csv', delimiter=',')
    data = (X, y, dataset)
    datasets.append(data)

In [4]:
class Module1Layer(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.dense0 = nn.Linear(n_features, n_neurons)
        self.act = nonlin
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.sigmoid(self.output(X))
        return X

In [5]:
class Module2Layers(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.dense0 = nn.Linear(n_features, n_neurons)
        self.act = nonlin
        self.dense1 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.act(self.dense1(X))
        X = self.sigmoid(self.output(X))
        return X

In [9]:
class Module3Layers(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.act = nonlin
        self.dense0 = nn.Linear(n_features, n_neurons)
        self.dense1 = nn.Linear(n_neurons, n_neurons)
        self.dense2 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.act(self.dense1(X))
        X = self.act(self.dense2(X))
        X = self.sigmoid(self.output(X))
        return X

In [6]:
class LogisticRegression(nn.Module):    
    # build the constructor
    def __init__(self, n_features=8):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)
    
    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

In [7]:
# Function to perform GridSearch on a single dataset
def run_grid_search(X, y, dataset_name, model, model_name):
    num_features = X.shape[1]

    clf = NeuralNetClassifier(
        model,
        criterion=nn.BCELoss,
        optimizer=optim.Adamax,
        max_epochs=100,
        batch_size=10,
        verbose=False
    )

    # define the grid search parameters
    param_grid = {
        'module__n_features': [num_features],
        'module__n_neurons': [16, 32, 64, 128, 256],
        'module__nonlin': [nn.Tanh(), nn.ReLU()],
        'batch_size': [16, 32, 64, 128],
        'max_epochs': [16, 32, 64, 128],
        'lr': [0.001, 0.01, 0.02]
    }

    X = torch.tensor(X.to_numpy(), dtype=torch.float32)
    y = torch.tensor(y.to_numpy(), dtype=torch.float32).reshape(-1, 1)

    grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, n_jobs=-1, cv=5, scoring='roc_auc')
    grid_result = grid_search.fit(X, y)

    #now we will Create and configure logger 
    logging.basicConfig(filename= dataset_name + "_" + model_name + ".log", 
    					format='%(asctime)s %(message)s', 
    					filemode='w') 

    #Let us Create an object 
    logger=logging.getLogger() 

    #Now we are going to Set the threshold of logger to DEBUG 
    logger.setLevel(logging.DEBUG)

    # summarize results
    logger.info("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
    logger.info("Best estimator configuration: \n %s" % (grid_result.best_estimator_))
    means = grid_result.cv_results_['mean_test_score']
    stds = grid_result.cv_results_['std_test_score']
    params = grid_result.cv_results_['params']
    for mean, stdev, param in zip(means, stds, params):
        logger.info("%f (%f) with: %r" % (mean, stdev, param))

    df = pd.DataFrame({'auc_mean': means, 'std_mean':stds, 'params':params})
    df2 = pd.concat([df, pd.DataFrame(list(df.params))], axis=1)
    df2.to_csv('./models/' + dataset_name + '/' + model_name +'.csv')    
    
    return grid_result.best_params_, grid_result.best_score_

In [8]:
# Execute GridSearch for each dataset in parallel
results = Parallel(n_jobs=-1)(delayed(run_grid_search)(X, y, dataset_name, Module1Layer, 'Module1Layer') for X, y, dataset_name in datasets)

In [ ]:
# Execute GridSearch for each dataset in parallel
results = Parallel(n_jobs=-1)(delayed(run_grid_search)(X, y, dataset_name, Module2Layers, 'Module2Layers') for X, y, dataset_name in datasets)

In [ ]:
# Execute GridSearch for each dataset in parallel
results = Parallel(n_jobs=-1)(delayed(run_grid_search)(X, y, dataset_name, Module2Layers, 'Module3Layers') for X, y, dataset_name in datasets)